# Ollama con GPU en Colab, para tu dashboard local

Deja Ollama corriendo aqui, con GPU, y publica una URL publica con `ngrok` para que tu dashboard (en tu portatil) le hable a este Ollama en vez de al tuyo local, que va por CPU.

**Antes de nada:** Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> Acelerador por hardware -> **GPU (T4)**.

**Necesitas una cuenta gratuita de ngrok** (https://ngrok.com -> Sign up), y tu authtoken personal, en Dashboard -> Your Authtoken. Es gratis, un minuto.

## Paso 1: instalar y arrancar Ollama

In [1]:
!curl -fsSL https://ollama.com/install.sh | sh

"sh" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [1]:
import os
import shutil
import subprocess
import time

import requests

OLLAMA_URL_LOCAL = "http://127.0.0.1:11434"

# Reutiliza un servidor que ya este escuchando para evitar levantar dos procesos.
try:
    respuesta_local = requests.get(f"{OLLAMA_URL_LOCAL}/api/tags", timeout=3)
except requests.RequestException:
    respuesta_local = None

if respuesta_local is not None and respuesta_local.ok:
    print("Ollama ya estaba activo en", OLLAMA_URL_LOCAL)
else:
    if shutil.which("ollama") is None:
        subprocess.run(
            ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
            check=True,
        )
        os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")

    ollama_bin = shutil.which("ollama")
    if ollama_bin is None:
        raise FileNotFoundError(
            "No se encontro Ollama despues de instalarlo. Ejecuta de nuevo esta celda."
        )

    entorno = os.environ.copy()
    entorno["OLLAMA_HOST"] = "127.0.0.1:11434"
    entorno["OLLAMA_ORIGINS"] = "*"

    proceso_ollama = subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=entorno,
    )

    for _ in range(30):
        if proceso_ollama.poll() is not None:
            salida = proceso_ollama.stdout.read() if proceso_ollama.stdout else ""
            raise RuntimeError(
                "Ollama termino antes de abrir el puerto 11434.\n"
                f"Salida del proceso:\n{salida}"
            )
        try:
            respuesta_local = requests.get(f"{OLLAMA_URL_LOCAL}/api/tags", timeout=2)
            if respuesta_local.ok:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    else:
        proceso_ollama.terminate()
        raise TimeoutError(
            "Ollama no respondio en http://127.0.0.1:11434 tras 30 segundos."
        )

    print("Ollama arrancado y comprobado en segundo plano (PID", proceso_ollama.pid, ")")

print("Endpoint local OK:", respuesta_local.status_code, f"{OLLAMA_URL_LOCAL}/api/tags")

Ollama ya estaba activo en http://127.0.0.1:11434
Endpoint local OK: 200 http://127.0.0.1:11434/api/tags


In [ ]:
!ollama pull llama3.2:3b

# Opcional: con GPU de verdad, qwen3:8b puede merecer la pena otra vez
# (en tu CPU local salio peor que llama3.2:3b: mas parametros, mas lento sin GPU.
# Con GPU la relacion puede invertirse -- se puede probar sin miedo.)
# !ollama pull qwen3:8b

In [ ]:
# Confirma que Ollama vera la GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Paso 2: publicar Ollama con ngrok
Pega tu authtoken de ngrok en la siguiente celda antes de ejecutarla.

In [ ]:
!pip install -q pyngrok

In [ ]:
import getpass
import os

import requests
from pyngrok import ngrok

NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN") or getpass.getpass(
    "Pega tu authtoken de ngrok (no se mostrara): "
)
if not NGROK_AUTHTOKEN:
    raise ValueError("Falta el authtoken de ngrok.")

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()
tunel = ngrok.connect(addr="127.0.0.1:11434", proto="http")
public_url = tunel.public_url.rstrip("/")
ngrok_headers = {"ngrok-skip-browser-warning": "true"}

try:
    respuesta_publica = requests.get(
        f"{public_url}/api/tags",
        headers=ngrok_headers,
        timeout=30,
    )
    respuesta_publica.raise_for_status()
except requests.RequestException as error:
    cuerpo = respuesta_publica.text[:500] if "respuesta_publica" in locals() else ""
    ngrok.disconnect(tunel.public_url)
    raise RuntimeError(
        f"La URL recien generada no alcanza Ollama: {public_url}/api/tags\n"
        f"HTTP: {getattr(respuesta_publica, 'status_code', 'sin respuesta')}\n"
        f"Respuesta: {cuerpo}\n{error}"
    ) from error

print("URL publica NUEVA de Ollama:", public_url)
print("Copia exactamente esta URL en el campo 'URL de Ollama remoto' del dashboard.")
print("Prueba de salud publica:", respuesta_publica.status_code)
print("Endpoint local que debe seguir activo: http://127.0.0.1:11434/api/tags")

## Paso 3: mantener esta sesión viva

Mientras quieras usar el dashboard local con esta GPU, **deja esta pestaña de Colab abierta** y no dejes que se quede inactiva demasiado tiempo (Colab gratuito desconecta sesiones inactivas tras ~90 minutos). La celda siguiente hace una comprobación periódica -- para de ejecutarla (botón de stop) cuando termines de usar el dashboard.

In [ ]:
import time

import requests

if "tunel" not in globals() or "public_url" not in globals():
    raise RuntimeError("Ejecuta primero la celda de ngrok para crear una public_url nueva.")

ngrok_headers = {"ngrok-skip-browser-warning": "true"}
print("Ollama activo en:", public_url)
print("Pulsa el boton de stop de esta celda cuando termines de usar el dashboard.")
print()

try:
    while True:
        time.sleep(60)
        local_ok = False
        public_ok = False
        try:
            local_ok = requests.get(
                "http://127.0.0.1:11434/api/tags", timeout=10
            ).ok
            public_ok = requests.get(
                f"{public_url}/api/tags",
                headers=ngrok_headers,
                timeout=30,
            ).ok
        except requests.RequestException:
            pass
        estado = "OK" if local_ok and public_ok else "ERROR"
        print(
            f"[{time.strftime('%H:%M:%S')}] {estado} | "
            f"local={local_ok} | publica={public_ok}",
            flush=True,
        )
except KeyboardInterrupt:
    print("\nDetenido.")

## Si la URL cambia

Cada vez que reinicies esta sesión de Colab, `ngrok.connect()` te da una URL **nueva** (el plan gratuito no permite una URL fija). Si reinicias, vuelve a ejecutar desde el Paso 2 y actualiza la URL en el dashboard.